<a href="https://colab.research.google.com/github/ssurapaneni34/702project/blob/main/mimic_lstm_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent-Augmented ICU Diagnosis Prediction
## LSTM Baseline — PyHealth Training Pipeline

**Plug-in point:** Once MIMIC-IV preprocessing is complete, set `DATA_PATH` below and run all cells.

---
### Expected Input Format (what the cleaned data should look like)

The preprocessing step should produce a single CSV file with **one row per ICU-hour**, shaped like:

| column | type | description |
|--------|------|-------------|
| `patient_id` | str/int | unique patient identifier |
| `stay_id` | str/int | unique ICU stay identifier |
| `hour` | int | hour index, 1–12 (first 12h of stay only) |
| `heart_rate` | float | beats per minute |
| `map` | float | mean arterial pressure (mmHg) |
| `temperature` | float | degrees Celsius |
| `respiratory_rate` | float | breaths per minute |
| `spo2` | float | oxygen saturation (%) |
| `creatinine` | float | mg/dL |
| `wbc` | float | white blood cell count (×10³/μL) |
| `urine_output` | float | mL/hour |
| `vasopressor` | int | binary flag (1 = on vasopressor) |
| `... (add more features as needed)` | float | any additional vitals/labs |
| `icd10_label` | int | integer class label (0 to K-1), primary discharge ICD-10 |
| `icd10_code` | str | original ICD-10 code string (e.g. 'A41.9') — keep for reference |
| `split` | str | 'train', 'val', or 'test' — assigned at PATIENT level |

**Key constraints your teammate should follow:**
- Every `stay_id` must have **exactly 12 rows** (hours 1–12). Use forward-fill then population mean for missing values.
- `icd10_label` should be 0-indexed integers for the **top-K most frequent** primary ICD-10 codes (recommend K=25–50). Filter out stays where the primary code falls outside top-K.
- The `split` column must be assigned at the **patient level** (all stays from one patient go to the same split). Recommended split: 70/10/20 train/val/test.
- Do NOT mix stays from the same patient across splits.

**Example:** A patient with `stay_id=1001` and 12 hours of data would appear as 12 rows (hours 1–12), all with `split='train'`, `icd10_label=3`, etc.

In [ ]:
# ── 0. Install dependencies (run once) ──────────────────────────────────────
# Uncomment if running for the first time
# !pip install pyhealth torch scikit-learn pandas numpy

In [ ]:
# ── 1. Config — SET THESE BEFORE RUNNING ────────────────────────────────────

DATA_PATH   = 'mimic_cleaned.csv'   # path to cleaned CSV described above
K           = 25                    # number of diagnosis classes (top-K ICD codes)
N_HOURS     = 12                    # sequence length
BATCH_SIZE  = 64
HIDDEN_DIM  = 128
N_LAYERS    = 2
LR          = 1e-3
MAX_EPOCHS  = 30
DEVICE      = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

# Feature columns — update this list to match whatever features your teammate extracted
FEATURE_COLS = [
    'heart_rate', 'map', 'temperature', 'respiratory_rate',
    'spo2', 'creatinine', 'wbc', 'urine_output', 'vasopressor'
]

print(f'Device: {DEVICE}, Features: {len(FEATURE_COLS)}, Classes: {K}')

In [ ]:
# ── 2. Load & validate data ──────────────────────────────────────────────────
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

# Quick sanity checks
assert 'stay_id'     in df.columns, 'Missing stay_id column'
assert 'icd10_label' in df.columns, 'Missing icd10_label column'
assert 'split'       in df.columns, 'Missing split column'
assert set(df['split'].unique()) <= {'train', 'val', 'test'}, 'split column must be train/val/test'

hours_per_stay = df.groupby('stay_id')['hour'].nunique()
bad = (hours_per_stay != N_HOURS).sum()
assert bad == 0, f'{bad} stays do not have exactly {N_HOURS} hourly rows — fix preprocessing'

print(f'Loaded {len(df)} rows | {df["stay_id"].nunique()} stays | {df["patient_id"].nunique()} patients')
print(df['split'].value_counts())
print(f'Label range: {df["icd10_label"].min()} – {df["icd10_label"].max()} (expected 0–{K-1})')

In [ ]:
# ── 3. Build sequence tensors ────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader

def build_tensors(df, feature_cols, n_hours):
    """Pivot long format → (n_stays, n_hours, n_features) tensor + labels."""
    stays = df.sort_values(['stay_id', 'hour'])
    groups = stays.groupby('stay_id', sort=False)

    X, y = [], []
    for _, grp in groups:
        X.append(grp[feature_cols].values.astype(np.float32))  # (12, n_features)
        y.append(int(grp['icd10_label'].iloc[0]))               # single label per stay

    return np.stack(X), np.array(y)  # (N, 12, F), (N,)


class ICUDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):              return len(self.y)
    def __getitem__(self, i):       return self.X[i], self.y[i]


splits = {}
for split in ['train', 'val', 'test']:
    X, y = build_tensors(df[df['split'] == split], FEATURE_COLS, N_HOURS)
    splits[split] = ICUDataset(X, y)
    print(f'{split}: {len(y)} stays')

loaders = {
    'train': DataLoader(splits['train'], batch_size=BATCH_SIZE, shuffle=True),
    'val':   DataLoader(splits['val'],   batch_size=BATCH_SIZE),
    'test':  DataLoader(splits['test'],  batch_size=BATCH_SIZE),
}

In [ ]:
# ── 4. Define LSTM model ─────────────────────────────────────────────────────
import torch.nn as nn

class LSTMDiagnosisClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers, n_classes, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        _, (h_n, _) = self.lstm(x)   # h_n: (n_layers, batch, hidden)
        out = self.fc(h_n[-1])        # last layer hidden state → logits
        return out                    # (batch, n_classes)

    def predict_topk(self, x, k=3):
        """Returns top-k class indices + softmax probabilities."""
        with torch.no_grad():
            logits = self.forward(x)
            probs  = torch.softmax(logits, dim=-1)
            topk   = torch.topk(probs, k, dim=-1)
        return topk.indices, topk.values  # (batch, k) each


model = LSTMDiagnosisClassifier(
    input_dim  = len(FEATURE_COLS),
    hidden_dim = HIDDEN_DIM,
    n_layers   = N_LAYERS,
    n_classes  = K
).to(DEVICE)

print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 5. Training loop with early stopping ────────────────────────────────────
import copy

optimizer    = torch.optim.Adam(model.parameters(), lr=LR)
criterion    = nn.CrossEntropyLoss()
best_val_acc = 0.0
best_weights = None
patience     = 5
no_improve   = 0

for epoch in range(1, MAX_EPOCHS + 1):
    # ── train
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in loaders['train']:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── validate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loaders['val']:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            preds = model(X_batch).argmax(dim=-1)
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
    val_acc = correct / total

    print(f'Epoch {epoch:02d} | train_loss={train_loss/len(loaders["train"]):.4f} | val_acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = copy.deepcopy(model.state_dict())
        no_improve   = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_weights)
print(f'\nBest val accuracy: {best_val_acc:.4f}')

In [ ]:
# ── 6. Evaluation on test set ────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

model.eval()
all_preds, all_labels, all_probs = [], [], []
top3_correct = 0

with torch.no_grad():
    for X_batch, y_batch in loaders['test']:
        X_batch = X_batch.to(DEVICE)
        logits  = model(X_batch)
        probs   = F.softmax(logits, dim=-1).cpu()
        top1    = probs.argmax(dim=-1)
        top3    = probs.topk(3, dim=-1).indices

        all_preds.extend(top1.numpy())
        all_labels.extend(y_batch.numpy())
        all_probs.extend(probs.numpy())

        # Top-3 accuracy (for context only — not a primary metric)
        for true, t3 in zip(y_batch.numpy(), top3.numpy()):
            top3_correct += int(true in t3)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

top1_acc = (all_preds == all_labels).mean()
top3_acc = top3_correct / len(all_labels)

try:
    auroc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
except ValueError as e:
    auroc = float('nan')
    print(f'AUROC could not be computed: {e}')

print(f'\n=== BASELINE LSTM RESULTS ===')
print(f'Top-1 Accuracy : {top1_acc:.4f}')
print(f'Top-3 Accuracy : {top3_acc:.4f}  (context only)')
print(f'Macro AUROC    : {auroc:.4f}')

In [ ]:
# ── 7. Save model + outputs for agent layer ──────────────────────────────────
# Save the model weights
torch.save(model.state_dict(), 'lstm_baseline.pt')
print('Model saved to lstm_baseline.pt')

# Save test predictions + top-3 candidates (agent layer will consume this)
test_df = df[df['split'] == 'test'].drop_duplicates('stay_id').reset_index(drop=True)
all_probs_tensor = torch.tensor(all_probs)
top3_indices = all_probs_tensor.topk(3, dim=-1).indices.numpy()
top3_scores  = all_probs_tensor.topk(3, dim=-1).values.numpy()

# Build a dataframe the agent layer can consume
agent_input_rows = []
for i, (stay_id, true_label) in enumerate(zip(test_df['stay_id'], all_labels)):
    agent_input_rows.append({
        'stay_id':        stay_id,
        'true_label':     true_label,
        'lstm_top1':      top3_indices[i][0],
        'lstm_top2':      top3_indices[i][1],
        'lstm_top3':      top3_indices[i][2],
        'conf_top1':      round(float(top3_scores[i][0]), 4),
        'conf_top2':      round(float(top3_scores[i][1]), 4),
        'conf_top3':      round(float(top3_scores[i][2]), 4),
    })

agent_df = pd.DataFrame(agent_input_rows)
agent_df.to_csv('lstm_agent_inputs.csv', index=False)
print('Agent layer inputs saved to lstm_agent_inputs.csv')
agent_df.head()

---
## What comes next (agent layer)

`lstm_agent_inputs.csv` contains one row per test patient with:
- The LSTM's top-3 candidate diagnosis labels + confidence scores
- The ground truth label

The **agent layer** (separate notebook) will:
1. For each row, pull the patient's 12-hour clinical summary from the cleaned data
2. Spin up 3 LLM agents (one per candidate) to generate clinical rationales via Gemini API
3. A ranking agent selects the final prediction
4. Compare `agent_final_prediction` vs `lstm_top1` vs `true_label`